In [1]:
# Load the processed HDFS dataset
import pandas as pd
from pathlib import Path

# Project paths
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load encoded sequences
encoded_df = pd.read_csv(
    PROCESSED_DIR / "encoded_sequences.csv"
)

# Load dataset split assignments
split_df = pd.read_csv(
    PROCESSED_DIR / "split_assignments.csv"
)

print("Encoded dataset shape:", encoded_df.shape)
print("\nSplit dataset shape:", split_df.shape)

print("\nEncoded dataset columns:")
print(encoded_df.columns.tolist())

print("\nSplit dataset columns:")
print(split_df.columns.tolist())

Encoded dataset shape: (575061, 4)

Split dataset shape: (575061, 3)

Encoded dataset columns:
['BlockId', 'EncodedFeatures', 'SequenceLength', 'ActualLabel']

Split dataset columns:
['BlockId', 'ActualLabel', 'Dataset']


In [2]:
# Validate dataset splits
# Check duplicate BlockIds
print("Duplicate BlockIds in encoded data:",
      encoded_df["BlockId"].duplicated().sum())

print("Duplicate BlockIds in split data:",
      split_df["BlockId"].duplicated().sum())

# Merge encoded data with split assignments
df = encoded_df.merge(
    split_df[["BlockId", "Dataset"]],
    on="BlockId",
    how="left",
    validate="one_to_one"
)

# Check missing split assignments
print("\nMissing split assignments:",
      df["Dataset"].isna().sum())

# Display split distribution
print("\nDataset split distribution:")
print(df["Dataset"].value_counts())

# Display labels within each split
print("\nLabel distribution by dataset:")
print(pd.crosstab(df["Dataset"], df["ActualLabel"]))

Duplicate BlockIds in encoded data: 0
Duplicate BlockIds in split data: 0

Missing split assignments: 0

Dataset split distribution:
Dataset
Train         391043
Test          100666
Validation     83352
Name: count, dtype: int64

Label distribution by dataset:
ActualLabel  Anomaly  Normal
Dataset                     
Test           16838   83828
Train              0  391043
Validation         0   83352


In [3]:
def extract_sequence_features(encoded_sequence):

    # Convert encoded string into a list of integers
    events = [int(x) for x in encoded_sequence.split(",")]

    # Basic sequence features
    sequence_length = len(events)
    unique_event_count = len(set(events))

    # Count changes between consecutive events
    event_change_count = sum(
        events[i] != events[i - 1]
        for i in range(1, sequence_length)
    )

    # Calculate event change ratio
    event_change_ratio = (
        event_change_count / (sequence_length - 1)
        if sequence_length > 1 else 0
    )

    # Create feature dictionary
    features = {
        "SequenceLength": sequence_length,
        "UniqueEventCount": unique_event_count,
        "EventChangeCount": event_change_count,
        "EventChangeRatio": event_change_ratio
    }

    # Count occurrences of all 29 event types
    for event_id in range(1, 30):
        features[f"E{event_id}_Count"] = events.count(event_id)

    return features

In [4]:
sample_sequence = encoded_df["EncodedFeatures"].iloc[0]

sample_features = extract_sequence_features(sample_sequence)

print("Original sequence:")
print(sample_sequence)

print("\nExtracted features:")
print(sample_features)

Original sequence:
5,22,5,5,11,9,11,9,11,9,26,26,26,4,3,4,2,23,23,23,21,21,21

Extracted features:
{'SequenceLength': 23, 'UniqueEventCount': 10, 'EventChangeCount': 15, 'EventChangeRatio': 0.6818181818181818, 'E1_Count': 0, 'E2_Count': 1, 'E3_Count': 1, 'E4_Count': 2, 'E5_Count': 3, 'E6_Count': 0, 'E7_Count': 0, 'E8_Count': 0, 'E9_Count': 3, 'E10_Count': 0, 'E11_Count': 3, 'E12_Count': 0, 'E13_Count': 0, 'E14_Count': 0, 'E15_Count': 0, 'E16_Count': 0, 'E17_Count': 0, 'E18_Count': 0, 'E19_Count': 0, 'E20_Count': 0, 'E21_Count': 3, 'E22_Count': 1, 'E23_Count': 3, 'E24_Count': 0, 'E25_Count': 0, 'E26_Count': 3, 'E27_Count': 0, 'E28_Count': 0, 'E29_Count': 0}


In [5]:
# Optimize feature extraction
from collections import Counter

def extract_sequence_features(encoded_sequence):

    events = [int(x) for x in encoded_sequence.split(",")]

    event_counts = Counter(events)

    sequence_length = len(events)
    unique_event_count = len(event_counts)

    event_change_count = sum(
        events[i] != events[i - 1]
        for i in range(1, sequence_length)
    )

    event_change_ratio = (
        event_change_count / (sequence_length - 1)
        if sequence_length > 1 else 0
    )

    features = {
        "SequenceLength": sequence_length,
        "UniqueEventCount": unique_event_count,
        "EventChangeCount": event_change_count,
        "EventChangeRatio": event_change_ratio
    }

    for event_id in range(1, 30):
        features[f"E{event_id}_Count"] = event_counts.get(event_id, 0)

    return features

In [6]:
# Test the optimized version
optimized_features = extract_sequence_features(sample_sequence)

print("Original and optimized results match:",
      sample_features == optimized_features)

print("Total extracted features:", len(optimized_features))

Original and optimized results match: True
Total extracted features: 33


In [8]:
# Extract features from all 575,061 sequences
import time

start_time = time.time()

# Extract features for every HDFS sequence
feature_records = [
    extract_sequence_features(sequence)
    for sequence in encoded_df["EncodedFeatures"]
]

# Convert extracted features into a DataFrame
features_df = pd.DataFrame(feature_records)

# Preserve BlockId and actual labels for later evaluation
features_df.insert(0, "BlockId", encoded_df["BlockId"].values)
features_df["ActualLabel"] = encoded_df["ActualLabel"].values

end_time = time.time()

print("Feature extraction completed!")
print("Dataset shape:", features_df.shape)
print("Total records:", len(features_df))
print("Total numerical features:", len(features_df.columns) - 2)
print("Time taken:", round(end_time - start_time, 2), "seconds")

print("\nFirst five records:")
display(features_df.head())

Feature extraction completed!
Dataset shape: (575061, 35)
Total records: 575061
Total numerical features: 33
Time taken: 40.98 seconds

First five records:


,BlockId,SequenceLength,UniqueEventCount,EventChangeCount,EventChangeRatio,E1_Count,E2_Count,E3_Count,E4_Count,E5_Count,...,E21_Count,E22_Count,E23_Count,E24_Count,E25_Count,E26_Count,E27_Count,E28_Count,E29_Count,ActualLabel
0,blk_7447720936221071962,23,10,15,0.681818,0,1,1,2,3,...,3,1,3,0,0,3,0,0,0,Normal
1,blk_-2129076755500825500,23,10,14,0.636364,0,1,1,2,3,...,3,1,3,0,0,3,0,0,0,Normal
2,blk_-9161574303807485051,22,9,14,0.666667,0,0,1,2,3,...,3,1,3,0,0,3,0,0,0,Normal
3,blk_1085930350461885394,22,9,12,0.571429,0,0,1,2,3,...,3,1,3,0,0,3,0,0,0,Normal
4,blk_1955612624060911018,23,10,15,0.681818,0,1,1,2,3,...,3,1,3,0,0,3,0,0,0,Normal


In [9]:
# Analyze feature distributions
# Select only numerical model features
feature_columns = [
    col for col in features_df.columns
    if col not in ["BlockId", "ActualLabel"]
]

X_all = features_df[feature_columns]

# Basic data quality checks
print("Missing values:", X_all.isnull().sum().sum())

print("Infinite values:", 
      (~X_all.apply(lambda col: col.abs() != float("inf"))).sum().sum())

# Identify constant features
constant_features = [
    col for col in feature_columns
    if X_all[col].nunique() <= 1
]

print("\nNumber of numerical features:", len(feature_columns))
print("Constant features:", constant_features)

# Feature statistics
display(X_all.describe().T)

Missing values: 0
Infinite values: 0

Number of numerical features: 33
Constant features: ['E22_Count']


,count,mean,std,min,25%,50%,75%,max
SequenceLength,575061.0,19.433815,5.177735,2.000000,19.000000,19.000000,20.000000,298.0
UniqueEventCount,575061.0,7.252293,1.578432,2.000000,7.000000,7.000000,8.000000,20.0
EventChangeCount,575061.0,11.332184,2.846250,1.000000,10.000000,11.000000,12.000000,84.0
EventChangeRatio,575061.0,0.624348,0.075498,0.063063,0.555556,0.611111,0.666667,1.0
E1_Count,575061.0,0.000017,0.005897,0.000000,0.000000,0.000000,0.000000,2.0
E2_Count,575061.0,0.208736,0.580644,0.000000,0.000000,0.000000,0.000000,4.0
E3_Count,575061.0,0.745531,2.517178,0.000000,0.000000,0.000000,0.000000,203.0
E4_Count,575061.0,0.619425,1.604554,0.000000,0.000000,0.000000,0.000000,41.0
E5_Count,575061.0,2.996607,0.220913,1.000000,3.000000,3.000000,3.000000,13.0
E6_Count,575061.0,0.012341,0.146923,0.000000,0.000000,0.000000,0.000000,10.0


In [10]:
# Prepare model features and dataset splits
# Remove constant features
model_features = [
    col for col in feature_columns
    if col not in constant_features
]

# Separate datasets
train_data = features_df.merge(
    split_df[["BlockId", "Dataset"]],
    on="BlockId",
    how="left"
)

# Create feature matrices
X_train = train_data.loc[
    train_data["Dataset"] == "Train", model_features
].copy()

X_val = train_data.loc[
    train_data["Dataset"] == "Validation", model_features
].copy()

X_test = train_data.loc[
    train_data["Dataset"] == "Test", model_features
].copy()

# Keep actual test labels separately
y_test = train_data.loc[
    train_data["Dataset"] == "Test", "ActualLabel"
].copy()

print("Features before removal:", len(feature_columns))
print("Features after removal:", len(model_features))

print("\nTraining shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Testing shape:", X_test.shape)

print("\nTest label distribution:")
print(y_test.value_counts())

Features before removal: 33
Features after removal: 32

Training shape: (391043, 32)
Validation shape: (83352, 32)
Testing shape: (100666, 32)

Test label distribution:
ActualLabel
Normal     83828
Anomaly    16838
Name: count, dtype: int64


In [11]:
# Train your first Isolation Forest model
from sklearn.ensemble import IsolationForest
import time

# Initialize Isolation Forest
isolation_forest = IsolationForest(
    n_estimators=100,
    max_samples=256,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

# Train using normal training records only
start_time = time.time()

isolation_forest.fit(X_train)

training_time = time.time() - start_time

print("Isolation Forest training completed!")
print("Number of trees:", len(isolation_forest.estimators_))
print("Training samples:", len(X_train))
print("Training time:", round(training_time, 2), "seconds")

Isolation Forest training completed!
Number of trees: 100
Training samples: 391043
Training time: 1.8 seconds


In [12]:
# Generate anomaly scores
train_scores_if = -isolation_forest.decision_function(X_train)

val_scores_if = -isolation_forest.decision_function(X_val)

test_scores_if = -isolation_forest.decision_function(X_test)

print("Anomaly scores generated successfully!")

print("\nTraining score statistics:")
print(pd.Series(train_scores_if).describe())

print("\nValidation score statistics:")
print(pd.Series(val_scores_if).describe())

print("\nTest score statistics:")
print(pd.Series(test_scores_if).describe())

Anomaly scores generated successfully!

Training score statistics:
count    391043.000000
mean         -0.060789
std           0.084174
min          -0.139937
25%          -0.132395
50%          -0.064329
75%           0.000973
max           0.305500
dtype: float64

Validation score statistics:
count    83352.000000
mean        -0.060979
std          0.084232
min         -0.139937
25%         -0.132395
50%         -0.064329
75%          0.000973
max          0.302172
dtype: float64

Test score statistics:
count    100666.000000
mean         -0.040038
std           0.095605
min          -0.139937
25%          -0.132395
50%          -0.028670
75%           0.015874
max           0.296827
dtype: float64


In [13]:
# Select an anomaly threshold
import numpy as np

# Calculate thresholds using normal validation scores
thresholds_if = {
    "95th_percentile": np.percentile(val_scores_if, 95),
    "97.5th_percentile": np.percentile(val_scores_if, 97.5),
    "99th_percentile": np.percentile(val_scores_if, 99)
}

print("Isolation Forest thresholds:")

for name, value in thresholds_if.items():
    print(f"{name}: {value:.6f}")

Isolation Forest thresholds:
95th_percentile: 0.094861
97.5th_percentile: 0.129145
99th_percentile: 0.180676


In [14]:
# Evaluate Isolation Forest predictions
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Convert actual labels into binary values
y_true_if = (y_test == "Anomaly").astype(int)

results_if = []

for name, threshold in thresholds_if.items():

    # Predict anomaly when score exceeds threshold
    y_pred_if = (test_scores_if > threshold).astype(int)

    precision = precision_score(y_true_if, y_pred_if)
    recall = recall_score(y_true_if, y_pred_if)
    f1 = f1_score(y_true_if, y_pred_if)

    tn, fp, fn, tp = confusion_matrix(
        y_true_if,
        y_pred_if
    ).ravel()

    results_if.append({
        "Threshold": name,
        "Threshold_Value": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

results_if_df = pd.DataFrame(results_if)

print(results_if_df.round(4).to_string(index=False))

        Threshold  Threshold_Value  Precision  Recall  F1_Score    TN   FP    FN   TP
  95th_percentile           0.0949     0.5255  0.2685    0.3554 79746 4082 12317 4521
97.5th_percentile           0.1291     0.6446  0.2244    0.3329 81744 2084 13059 3779
  99th_percentile           0.1807     0.7449  0.1487    0.2478 82971  857 14335 2503


In [15]:
# Calculate ROC-AUC and PR-AUC
from sklearn.metrics import roc_auc_score, average_precision_score

# Actual binary labels
y_true_if = (y_test == "Anomaly").astype(int)

# Ranking metrics using continuous anomaly scores
roc_auc_if = roc_auc_score(y_true_if, test_scores_if)

pr_auc_if = average_precision_score(
    y_true_if,
    test_scores_if
)

print(f"Isolation Forest ROC-AUC: {roc_auc_if:.4f}")
print(f"Isolation Forest PR-AUC:  {pr_auc_if:.4f}")

Isolation Forest ROC-AUC: 0.8528
Isolation Forest PR-AUC:  0.5258


In [16]:
# Save the Isolation Forest model
import joblib
from pathlib import Path

# Project directories
PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save model and its required configuration together
isolation_forest_artifact = {
    "model": isolation_forest,
    "feature_columns": model_features,
    "thresholds": thresholds_if,
    "selected_threshold": thresholds_if["97.5th_percentile"],
    "model_config": {
        "n_estimators": 100,
        "max_samples": 256,
        "contamination": "auto",
        "random_state": 42
    }
}

model_path = MODEL_DIR / "isolation_forest_model.joblib"

joblib.dump(isolation_forest_artifact, model_path)

print("Isolation Forest model saved successfully!")
print("Model path:", model_path)
print("Number of features:", len(model_features))
print("Selected threshold:", isolation_forest_artifact["selected_threshold"])

Isolation Forest model saved successfully!
Model path: D:\Srinath Folder\HDFS_Log_Anomaly_Detection_Project_Structure\HDFS_Log_Anomaly_Detection\models\isolation_forest_model.joblib
Number of features: 32
Selected threshold: 0.12914456293378518


In [17]:
# Verify the saved model
import joblib
import numpy as np

# Load saved artifact
loaded_artifact = joblib.load(model_path)

loaded_model = loaded_artifact["model"]
loaded_features = loaded_artifact["feature_columns"]
loaded_threshold = loaded_artifact["selected_threshold"]

# Generate scores using original and reloaded models
original_scores = -isolation_forest.decision_function(
    X_test.iloc[:100]
)

reloaded_scores = -loaded_model.decision_function(
    X_test.iloc[:100][loaded_features]
)

# Compare predictions
max_difference = np.max(
    np.abs(original_scores - reloaded_scores)
)

print("Model reload successful!")
print("Number of loaded features:", len(loaded_features))
print("Loaded threshold:", loaded_threshold)
print("Maximum score difference:", max_difference)

Model reload successful!
Number of loaded features: 32
Loaded threshold: 0.12914456293378518
Maximum score difference: 0.0


In [18]:
# Prepare a unified model comparison
model_comparison = pd.DataFrame([
    {
        "Model": "LSTM Autoencoder",
        "ROC_AUC": 0.992409,
        "PR_AUC": 0.957942,
        "Precision": 0.8823,
        "Recall": 0.9241,
        "F1_Score": 0.9027
    },
    {
        "Model": "Isolation Forest",
        "ROC_AUC": roc_auc_if,
        "PR_AUC": pr_auc_if,
        "Precision": 0.6446,
        "Recall": 0.2244,
        "F1_Score": 0.3329
    }
])

print("Model Performance Comparison")
display(model_comparison.round(4))

Model Performance Comparison


,Model,ROC_AUC,PR_AUC,Precision,Recall,F1_Score
0,LSTM Autoencoder,0.9924,0.9579,0.8823,0.9241,0.9027
1,Isolation Forest,0.8528,0.5258,0.6446,0.2244,0.3329


In [19]:
# Investigate why Isolation Forest underperforms
# Separate normal and anomaly records in the test set
normal_test = features_df[
    features_df["BlockId"].isin(
        train_data.loc[
            train_data["Dataset"] == "Test", "BlockId"
        ]
    ) &
    (features_df["ActualLabel"] == "Normal")
]

anomaly_test = features_df[
    features_df["BlockId"].isin(
        train_data.loc[
            train_data["Dataset"] == "Test", "BlockId"
        ]
    ) &
    (features_df["ActualLabel"] == "Anomaly")
]

# Compare selected feature averages
selected_features = [
    "SequenceLength",
    "UniqueEventCount",
    "EventChangeCount",
    "EventChangeRatio",
    "E3_Count",
    "E5_Count",
    "E9_Count",
    "E11_Count",
    "E21_Count",
    "E23_Count",
    "E26_Count"
]

feature_comparison = pd.DataFrame({
    "Normal_Mean": normal_test[selected_features].mean(),
    "Anomaly_Mean": anomaly_test[selected_features].mean()
})

feature_comparison["Difference"] = (
    feature_comparison["Anomaly_Mean"] -
    feature_comparison["Normal_Mean"]
)

display(feature_comparison.round(3))

,Normal_Mean,Anomaly_Mean,Difference
SequenceLength,19.503,17.119,-2.384
UniqueEventCount,7.258,7.159,-0.099
EventChangeCount,11.348,11.007,-0.341
EventChangeRatio,0.621,0.742,0.121
E3_Count,0.743,0.703,-0.040
E5_Count,3.005,2.729,-0.276
E9_Count,3.000,1.891,-1.109
E11_Count,3.000,1.901,-1.099
E21_Count,2.447,2.107,-0.341
E23_Count,2.443,1.896,-0.547


In [20]:
# Identify the strongest individual features
import numpy as np

normal_std = normal_test[model_features].std()
anomaly_std = anomaly_test[model_features].std()

normal_n = len(normal_test)
anomaly_n = len(anomaly_test)

pooled_std = np.sqrt(
    (
        (normal_n - 1) * normal_std**2
        + (anomaly_n - 1) * anomaly_std**2
    )
    / (normal_n + anomaly_n - 2)
)

cohens_d = (
    anomaly_test[model_features].mean()
    - normal_test[model_features].mean()
) / pooled_std.replace(0, np.nan)

feature_effects = (
    cohens_d.abs()
    .sort_values(ascending=False)
    .rename("Absolute_Cohens_d")
    .to_frame()
)

display(feature_effects.head(15).round(3))

,Absolute_Cohens_d
E9_Count,1.874
E11_Count,1.851
E20_Count,1.571
EventChangeRatio,1.404
E7_Count,1.162
E18_Count,1.055
E25_Count,1.055
E16_Count,1.051
E6_Count,1.035
E26_Count,0.720


In [21]:
# Check feature correlation and redundancy
# Calculate feature correlation using training data only
correlation_matrix = X_train.corr(method="spearman")

# Extract upper triangle to avoid duplicate pairs
upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

# Find feature pairs with absolute correlation above 0.80
high_correlations = (
    upper_triangle
    .stack()
    .reset_index()
)

high_correlations.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

high_correlations["Absolute_Correlation"] = (
    high_correlations["Correlation"].abs()
)

high_correlations = high_correlations.sort_values(
    "Absolute_Correlation",
    ascending=False
)

print("Highly correlated feature pairs (|correlation| > 0.80):")

display(
    high_correlations[
        high_correlations["Absolute_Correlation"] > 0.80
    ].round(3)
)

Highly correlated feature pairs (|correlation| > 0.80):


,Feature_1,Feature_2,Correlation,Absolute_Correlation
699,E18_Count,E25_Count,1.000,1.000
265,E5_Count,E6_Count,1.000,1.000
275,E5_Count,E16_Count,1.000,1.000
307,E6_Count,E16_Count,1.000,1.000
629,E16_Count,E18_Count,1.000,1.000
635,E16_Count,E25_Count,1.000,1.000
277,E5_Count,E18_Count,1.000,1.000
283,E5_Count,E25_Count,1.000,1.000
315,E6_Count,E25_Count,1.000,1.000
309,E6_Count,E18_Count,1.000,1.000


In [22]:
# Identify redundant feature groups
# Filter strongly correlated feature pairs
redundant_pairs = high_correlations[
    high_correlations["Absolute_Correlation"] >= 0.95
].copy()

print("Strongly correlated feature pairs:")
display(redundant_pairs.round(4))

print("\nNumber of strongly correlated pairs:",
      len(redundant_pairs))

Strongly correlated feature pairs:


,Feature_1,Feature_2,Correlation,Absolute_Correlation
699,E18_Count,E25_Count,1.0000,1.0000
265,E5_Count,E6_Count,1.0000,1.0000
275,E5_Count,E16_Count,1.0000,1.0000
307,E6_Count,E16_Count,1.0000,1.0000
629,E16_Count,E18_Count,0.9996,0.9996
635,E16_Count,E25_Count,0.9996,0.9996
277,E5_Count,E18_Count,0.9996,0.9996
283,E5_Count,E25_Count,0.9996,0.9996
315,E6_Count,E25_Count,0.9996,0.9996
309,E6_Count,E18_Count,0.9996,0.9996



Number of strongly correlated pairs: 18


In [23]:
# Run a controlled feature-reduction experiment
# Identify highly correlated feature pairs using training data only

redundant_pairs = high_correlations[
    high_correlations["Absolute_Correlation"] >= 0.95
].copy()

print("Features involved in strong correlations:")

correlated_features = sorted(
    set(redundant_pairs["Feature_1"]).union(
        set(redundant_pairs["Feature_2"])
    )
)

print(correlated_features)
print("\nNumber of unique features involved:", len(correlated_features))

Features involved in strong correlations:
['E16_Count', 'E18_Count', 'E21_Count', 'E23_Count', 'E25_Count', 'E26_Count', 'E3_Count', 'E4_Count', 'E5_Count', 'E6_Count', 'SequenceLength', 'UniqueEventCount']

Number of unique features involved: 12


In [24]:
# Build the reduced feature list
# Start with the original feature columns
reduced_feature_columns = model_features.copy()

# Remove selected redundant event-count features
features_to_remove = [
    "E6_Count",
    "E16_Count",
    "E18_Count",
    "E25_Count",
    "E26_Count",
    "E23_Count",
    "E4_Count"
]

reduced_feature_columns = [
    feature for feature in reduced_feature_columns
    if feature not in features_to_remove
]

print("Original feature count:", len(model_features))
print("Reduced feature count:", len(reduced_feature_columns))

print("\nFeatures retained:")
print(reduced_feature_columns)

print("\nFeatures removed:")
print(features_to_remove)

Original feature count: 32
Reduced feature count: 25

Features retained:
['SequenceLength', 'UniqueEventCount', 'EventChangeCount', 'EventChangeRatio', 'E1_Count', 'E2_Count', 'E3_Count', 'E5_Count', 'E7_Count', 'E8_Count', 'E9_Count', 'E10_Count', 'E11_Count', 'E12_Count', 'E13_Count', 'E14_Count', 'E15_Count', 'E17_Count', 'E19_Count', 'E20_Count', 'E21_Count', 'E24_Count', 'E27_Count', 'E28_Count', 'E29_Count']

Features removed:
['E6_Count', 'E16_Count', 'E18_Count', 'E25_Count', 'E26_Count', 'E23_Count', 'E4_Count']


In [25]:
# Create reduced datasets
# Create reduced feature matrices

X_train_reduced = X_train[reduced_feature_columns].copy()
X_val_reduced = X_val[reduced_feature_columns].copy()
X_test_reduced = X_test[reduced_feature_columns].copy()

print("Reduced training shape:", X_train_reduced.shape)
print("Reduced validation shape:", X_val_reduced.shape)
print("Reduced test shape:", X_test_reduced.shape)

Reduced training shape: (391043, 25)
Reduced validation shape: (83352, 25)
Reduced test shape: (100666, 25)


In [26]:
# Train the reduced Isolation Forest
from sklearn.ensemble import IsolationForest
import time

# Initialize reduced-feature model
isolation_forest_reduced = IsolationForest(
    n_estimators=100,
    max_samples=256,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

# Train using normal training data only
start_time = time.time()

isolation_forest_reduced.fit(X_train_reduced)

training_time_reduced = time.time() - start_time

print(f"Reduced model training time: {training_time_reduced:.2f} seconds")
print("Number of features:", isolation_forest_reduced.n_features_in_)

Reduced model training time: 0.83 seconds
Number of features: 25


In [27]:
# Generate anomaly scores
# Higher score = more anomalous

train_scores_if_reduced = -isolation_forest_reduced.decision_function(
    X_train_reduced
)

val_scores_if_reduced = -isolation_forest_reduced.decision_function(
    X_val_reduced
)

test_scores_if_reduced = -isolation_forest_reduced.decision_function(
    X_test_reduced
)

print("Training scores:", train_scores_if_reduced.shape)
print("Validation scores:", val_scores_if_reduced.shape)
print("Test scores:", test_scores_if_reduced.shape)

print("\nValidation score statistics:")
print("Mean:", val_scores_if_reduced.mean())
print("Std:", val_scores_if_reduced.std())
print("Min:", val_scores_if_reduced.min())
print("Max:", val_scores_if_reduced.max())

Training scores: (391043,)
Validation scores: (83352,)
Test scores: (100666,)

Validation score statistics:
Mean: -0.056013316608544875
Std: 0.08727476687958795
Min: -0.13865305451901389
Max: 0.24861897149615964


In [28]:
# Calculate threshold using normal validation data
threshold_if_reduced = np.percentile(
    val_scores_if_reduced, 97.5
)

print("Reduced IF threshold (97.5th percentile):",
      threshold_if_reduced)

Reduced IF threshold (97.5th percentile): 0.14169469402267054


In [30]:
# Convert actual labels into binary format
# Normal = 0, Anomaly = 1

y_test_binary = (
    pd.Series(y_test)
    .map({"Normal": 0, "Anomaly": 1})
    .to_numpy()
)

print("Actual test labels:")
print(pd.Series(y_test_binary).value_counts())

print("\nPredicted test labels:")
print(pd.Series(y_pred_if_reduced).value_counts())

Actual test labels:
0    83828
1    16838
Name: count, dtype: int64

Predicted test labels:
0    95651
1     5015
Name: count, dtype: int64


In [31]:
# Evaluate reduced Isolation Forest

precision_reduced = precision_score(
    y_test_binary, y_pred_if_reduced
)

recall_reduced = recall_score(
    y_test_binary, y_pred_if_reduced
)

f1_reduced = f1_score(
    y_test_binary, y_pred_if_reduced
)

roc_auc_reduced = roc_auc_score(
    y_test_binary, test_scores_if_reduced
)

pr_auc_reduced = average_precision_score(
    y_test_binary, test_scores_if_reduced
)

print("REDUCED ISOLATION FOREST RESULTS")
print("--------------------------------")
print(f"Precision: {precision_reduced:.4f}")
print(f"Recall:    {recall_reduced:.4f}")
print(f"F1-score:  {f1_reduced:.4f}")
print(f"ROC-AUC:   {roc_auc_reduced:.4f}")
print(f"PR-AUC:    {pr_auc_reduced:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_binary, y_pred_if_reduced))

REDUCED ISOLATION FOREST RESULTS
--------------------------------
Precision: 0.5645
Recall:    0.1681
F1-score:  0.2591
ROC-AUC:   0.8700
PR-AUC:    0.5441

Confusion Matrix:
[[81644  2184]
 [14007  2831]]


In [32]:
# Save the experiment results
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
output_dir = project_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

comparison_df = pd.DataFrame([
    {
        "Model": "LSTM Autoencoder",
        "Features": "Event sequences",
        "Precision": 0.8823,
        "Recall": 0.9241,
        "F1": 0.9027,
        "ROC_AUC": 0.9924,
        "PR_AUC": 0.9579
    },
    {
        "Model": "Isolation Forest Original",
        "Features": 32,
        "Precision": 0.6446,
        "Recall": 0.2244,
        "F1": 0.3329,
        "ROC_AUC": 0.8528,
        "PR_AUC": 0.5258
    },
    {
        "Model": "Isolation Forest Reduced",
        "Features": 25,
        "Precision": precision_reduced,
        "Recall": recall_reduced,
        "F1": f1_reduced,
        "ROC_AUC": roc_auc_reduced,
        "PR_AUC": pr_auc_reduced
    }
])

output_path = output_dir / "final_model_comparison.csv"
comparison_df.to_csv(output_path, index=False)

print("Saved:", output_path)
display(comparison_df.round(4))

Saved: D:\Srinath Folder\HDFS_Log_Anomaly_Detection_Project_Structure\HDFS_Log_Anomaly_Detection\outputs\final_model_comparison.csv


,Model,Features,Precision,Recall,F1,ROC_AUC,PR_AUC
0,LSTM Autoencoder,Event sequences,0.8823,0.9241,0.9027,0.9924,0.9579
1,Isolation Forest Original,32,0.6446,0.2244,0.3329,0.8528,0.5258
2,Isolation Forest Reduced,25,0.5645,0.1681,0.2591,0.8700,0.5441
